# 36. Transformer blocks and positional information

**Level:** Advanced

**Prerequisites:** 35: attention and masking.

[Course index](00_START_HERE.ipynb) | [Previous](35_Attention_and_Causal_Masking.ipynb) | [Next](37_Tokens_Subwords_and_Context_Windows.ipynb)

**Topics:** Transformers, Transformer blocks, Multi-head attention, Positional information, Residual connections, Layer normalization, Encoder vs decoder.

**Matching script:** `examples\36_Transformer_Blocks_and_Positions.py`

**How to study:** read the notes, predict an example's output, run it, trace each change in data/shape/state, then complete the practice task. Every worked code cell contains its own imports and data and can run independently of other notebooks.

**Execution contract:** Python 3.11+, course dependencies, CPU only, synthetic/local data. No hosted API requests, model-weight downloads, dataset downloads, listening web servers, credentials, or real device changes. The temporary-file lesson cleans up its own files. Expected errors are caught narrowly and asserted; unexpected errors must remain visible.

## Concept notes and mental model

A transformer block combines attention with a positionwise feed-forward network, residual
connections, and normalization. Positional information is necessary because self-attention
alone does not identify token order in the way a recurrent traversal does.

Positional schemes include learned absolute embeddings, sinusoidal features, and relative or
rotary representations. They are not equivalent, and extending a context limit is not simply
appending more positions to any model. Architecture and training determine usable behavior.

Encoder-only models commonly use bidirectional context for representations. Decoder-only
causal models predict continuations. Encoder-decoder models connect an input encoder to an
autoregressive decoder with cross-attention. Naming depends on architecture, not just whether
a library class happens to contain 'Encoder' in its name.

In this example a TransformerEncoderLayer is given a causal mask to demonstrate a decoder-like
masked block. It is not a complete pretrained GPT model. Dropout is disabled for the invariant
check, and future-token perturbation verifies the mask rather than trusting a shape alone.

## Mathematics or data contract

Residual update: h_next = h + sublayer(h), with normalization placement depending on the architecture.

## Important limitation

An untrained small block can verify shapes and causality, not language quality.

Distinguish what is implemented and asserted below from what would need additional data, training, independent evaluation, authorization, or operational controls.

## Worked example 1: Check multi-head block causality with a perturbation

**Purpose:** Demonstrate learned projections and positional embeddings.

### Read the code in this order
1. Build token plus position embeddings.
2. Create a two-head transformer layer with dropout zero.
3. Apply an upper-triangular causal mask.
4. Change the last token and assert earlier outputs remain unchanged.

### Syntax and behavior to notice
- `assert` checks the stated behavior; a failed check raises AssertionError.
- A fixed random seed makes the chosen sampling reproducible within the environment.
- `with`/`async with` bounds a resource lifetime or a temporary execution mode.

### Expected output
This is the output captured from executing the worked example in the recorded environment. Package versions and floating-point rounding can change numeric details; the assertions express the required shapes, invariants, and thresholds.

```text
block shape: (1, 5, 8) earlier positions unchanged
```

**Before running:** identify the input variables, which values are learned versus chosen, and what each assertion proves. A successful shape test does not by itself prove predictive quality. Training-only loss checks are labeled as such.

In [1]:
import torch
from torch import nn

torch.set_num_threads(1)
torch.manual_seed(42)
token_embedding, position_embedding = nn.Embedding(12, 8), nn.Embedding(5, 8)
tokens = torch.tensor([[1, 2, 3, 4, 5]])
positions = torch.arange(tokens.shape[1])
layer = nn.TransformerEncoderLayer(d_model=8, nhead=2, dim_feedforward=16,
                                   dropout=0.0, batch_first=True)
layer.eval()
mask = torch.triu(torch.ones(5, 5, dtype=torch.bool), diagonal=1)
with torch.inference_mode():
    original = token_embedding(tokens) + position_embedding(positions)
    output = layer(original, src_mask=mask)
    changed_tokens = tokens.clone()
    changed_tokens[0, -1] = 9
    changed = token_embedding(changed_tokens) + position_embedding(positions)
    perturbed = layer(changed, src_mask=mask)
assert output.shape == (1, 5, 8)
torch.testing.assert_close(output[:, :-1], perturbed[:, :-1], atol=1e-6, rtol=1e-5)
assert not torch.allclose(output[:, -1], perturbed[:, -1])
print("block shape:", tuple(output.shape), "earlier positions unchanged")

block shape: (1, 5, 8) earlier positions unchanged


### Interpret example 1

The result should demonstrate this purpose: **Demonstrate learned projections and positional embeddings.**

Follow the assertions back to the exact data or state being checked. Where a model was fitted, distinguish training fit from held-out evaluation. Where input is rejected, inspect whether rejection happens before any intended mutation or tool execution. Do not delete a failed assertion to make an incorrect result look valid.

## Practice: extend the concept yourself

Add a feed-forward projection and output vocabulary head, then count trainable parameters.

1. Write your expected result before editing code.
2. Add one normal-case test and one boundary/failure-case test.
3. Keep training/validation/test responsibilities separate where relevant.
4. Report an honest failure or unknown result rather than a success-shaped fallback.

The task is intentionally not filled in for you; the worked examples above are complete and runnable. Add your own code cell for the extension.

## Understanding checkpoint
- Explain `Transformers` in your own words and give one example of when it is useful.
- Trace one worked example from input through transformation to returned or printed output.
- State this lesson's mathematical/data contract: Residual update: h_next = h + sublayer(h), with normalization placement depending on the architecture.
- Name the boundary of the demonstration: An untrained small block can verify shapes and causality, not language quality.
- Describe one test that could catch a plausible but wrong implementation.

## Primary references and further reading

- `https://arxiv.org/abs/1706.03762`
- `https://docs.pytorch.org/docs/stable/generated/torch.nn.TransformerEncoderLayer.html`

Research snapshot: **2026-10-17**. Foundational references need not be recent. Mutable library docs describe their current version; compare with the tested dependency versions in the course manifest. References are reading material, not runtime downloads.